# Calibration Analysis — Notebook 11
## VLM Medical VQA Benchmark

**Goal:** Measure not just *what* models predict on closed (Yes/No) questions, but *how
confident* they are. Confidence is derived from the softmax probability over the model's
full vocabulary at the first generated token position.

**Runs (change `RUN_CONFIG` in Cell 1):**

| Run | Config | Model | Dataset | Est. time |
|---|---|---|---|---|
| A | `medgemma_slake` | MedGemma-4B | SLAKE (~355 Qs) | ~10 min |
| B | `medgemma_vqa_rad` | MedGemma-4B | VQA-RAD (~251 Qs) | ~6 min |
| C | `gemma3_slake` | Gemma-3-4B | SLAKE (~355 Qs) | ~10 min |
| D | `gemma3_vqa_rad` | Gemma-3-4B | VQA-RAD (~251 Qs) | ~6 min |

Download each output `.jsonl` and place in `outputs/_archive/calibration/`, then run
`python3 scripts/calibration_analysis.py` locally.

## Cell 1 — Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  SET THIS BEFORE RUNNING
# ══════════════════════════════════════════════════════════════════
RUN_CONFIG = "medgemma_slake"   # medgemma_slake | medgemma_vqa_rad | gemma3_slake | gemma3_vqa_rad
# ══════════════════════════════════════════════════════════════════

import os

# SLAKE images will be downloaded and extracted here (only used for SLAKE runs)
SLAKE_IMGS_DIR = '/kaggle/working/slake_imgs'

CONFIGS = {
    "medgemma_slake": {
        "model_id":    "google/medgemma-4b-it",
        "model_short": "MedGemma-4B",
        "dataset":     "slake",
        "use_4bit":    False,
        "file_tag":    "medgemma_4b__slake__calibration",
    },
    "medgemma_vqa_rad": {
        "model_id":    "google/medgemma-4b-it",
        "model_short": "MedGemma-4B",
        "dataset":     "vqa_rad",
        "use_4bit":    False,
        "file_tag":    "medgemma_4b__vqa_rad__calibration",
    },
    "gemma3_slake": {
        "model_id":    "google/gemma-3-4b-it",
        "model_short": "Gemma-3-4B",
        "dataset":     "slake",
        "use_4bit":    False,
        "file_tag":    "gemma3_4b__slake__calibration",
    },
    "gemma3_vqa_rad": {
        "model_id":    "google/gemma-3-4b-it",
        "model_short": "Gemma-3-4B",
        "dataset":     "vqa_rad",
        "use_4bit":    False,
        "file_tag":    "gemma3_4b__vqa_rad__calibration",
    },
}

cfg         = CONFIGS[RUN_CONFIG]
MODEL_ID    = cfg["model_id"]
MODEL_SHORT = cfg["model_short"]
DATASET     = cfg["dataset"]
USE_4BIT    = cfg["use_4bit"]
FILE_TAG    = cfg["file_tag"]
OUTPUT_DIR  = "/kaggle/working"
OUT_PATH    = f"{OUTPUT_DIR}/{FILE_TAG}.jsonl"

print(f"Config  : {RUN_CONFIG}")
print(f"Model   : {MODEL_ID}")
print(f"Dataset : {DATASET}")
print(f"Output  : {OUT_PATH}")

## Cell 2 — GPU Check + Install

In [ ]:
!nvidia-smi
!pip install -q transformers==4.51.3 accelerate bitsandbytes datasets tqdm

## Cell 3 — Download SLAKE Images
SLAKE images are **not** embedded in the HuggingFace dataset — they must be downloaded
separately from `imgs.zip`. Skip this cell for VQA-RAD runs (images are embedded there).

In [ ]:
import zipfile, subprocess

if DATASET == 'slake':
    os.makedirs(SLAKE_IMGS_DIR, exist_ok=True)
    zip_path = f'{SLAKE_IMGS_DIR}/imgs.zip'
    url = 'https://huggingface.co/datasets/BoKelvin/SLAKE/resolve/main/imgs.zip'

    if not os.path.exists(zip_path):
        print('Downloading SLAKE images ...')
        subprocess.run(['curl', '-L', url, '-o', zip_path], check=True)
        print('Download complete.')
    else:
        print('Zip already present, skipping download.')

    print('Extracting ...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(SLAKE_IMGS_DIR)

    # Count images to verify extraction
    imgs = []
    for root, dirs, files in os.walk(SLAKE_IMGS_DIR):
        for f in files:
            if f.endswith(('.jpg', '.png', '.jpeg')):
                imgs.append(os.path.join(root, f))
    print(f'Extracted {len(imgs)} images.')

    # The zip extracts to slake_imgs/imgs/<subfolder>/source.jpg
    # img_name in SLAKE metadata is like 'xmlab102/source.jpg'
    SLAKE_IMG_BASE = os.path.join(SLAKE_IMGS_DIR, 'imgs')
    if not os.path.isdir(SLAKE_IMG_BASE):
        SLAKE_IMG_BASE = SLAKE_IMGS_DIR  # fallback if zip extracted flat

    print(f'Image base : {SLAKE_IMG_BASE}')
    print(f'Exists     : {os.path.isdir(SLAKE_IMG_BASE)}')
    for p in imgs[:3]:
        print(f'  {p}')
else:
    SLAKE_IMG_BASE = None
    print('VQA-RAD run — images are embedded in the dataset, no download needed.')

## Cell 4 — Imports

In [ ]:
import json
import torch
import torch.nn.functional as F
from PIL import Image
from datasets import load_dataset
from tqdm import tqdm
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {device}')
print(f'PyTorch : {torch.__version__}')
if device == 'cuda':
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}  {p.total_memory // 1024**2} MB')

## Cell 5 — HuggingFace Authentication
Required for `google/medgemma-4b-it`. Add `HF_TOKEN` to Kaggle Secrets.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
try:
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=hf_token)
    print('HF login successful.')
except Exception as e:
    print(f'HF login skipped: {e}')

## Cell 6 — Load Dataset and Filter to Closed (Yes/No) Questions

**SLAKE:** Images are loaded from disk using `img_name` (e.g. `xmlab102/source.jpg`)
resolved against `SLAKE_IMG_BASE`.

**VQA-RAD:** Images are embedded directly in the HuggingFace record as PIL objects.

In [ ]:
if DATASET == 'slake':
    ds_raw = load_dataset('BoKelvin/SLAKE')
    test_all = [s for s in ds_raw['test'] if s.get('q_lang') == 'en']

    def get_image(sample):
        img_name = sample.get('img_name', '')
        img_path = os.path.join(SLAKE_IMG_BASE, img_name)
        return Image.open(img_path).convert('RGB')

    def get_question(sample):
        return str(sample.get('question', '')).strip()

    def get_ground_truth(sample):
        return str(sample.get('answer', '')).strip().lower()

    def is_closed_q(sample):
        return str(sample.get('answer_type', '')).upper() == 'CLOSED'

    records = [s for s in test_all if is_closed_q(s)]
    print(f'SLAKE EN closed test questions : {len(records)}')

    # Verify image loading works
    test_img = get_image(records[0])
    print(f'Image load OK — size={test_img.size}, mode={test_img.mode}')

elif DATASET == 'vqa_rad':
    ds_raw = load_dataset('flaviagiammarino/vqa-rad')
    test_all = list(ds_raw['test'])

    def get_image(sample):
        # Images are embedded as PIL objects in this dataset
        img = sample.get('image')
        if img is None:
            raise ValueError('No image field — check dataset.')
        return img.convert('RGB')

    def get_question(sample):
        return str(sample.get('question', '')).strip()

    def get_ground_truth(sample):
        return str(sample.get('answer', '')).strip().lower()

    def is_closed_q(sample):
        return str(sample.get('answer_type', '')).upper() == 'CLOSED'

    records = [s for s in test_all if is_closed_q(s)]
    print(f'VQA-RAD closed test questions : {len(records)}')

    test_img = get_image(records[0])
    print(f'Image load OK — size={test_img.size}, mode={test_img.mode}')

print(f'\nSample — Q: {get_question(records[0])}')
print(f'          GT: {get_ground_truth(records[0])}')

## Cell 7 — Load Model and Processor

In [ ]:
dtype = torch.bfloat16 if device == 'cuda' else torch.float32

if USE_4BIT:
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, quantization_config=bnb_cfg, device_map='auto', trust_remote_code=True
    )
else:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, torch_dtype=dtype, device_map='auto', trust_remote_code=True
    )

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model.eval()
param_b = sum(p.numel() for p in model.parameters()) / 1e9
print(f'Model loaded : {MODEL_ID}  ({param_b:.2f}B params)')

## Cell 8 — Resolve Yes / No Token IDs

We extract P(Yes) and P(No) from the softmax distribution at the first generated token.
We collect all token IDs that decode to 'Yes', 'yes', 'No', 'no' (including SentencePiece
leading-space variants) to handle tokenizer differences between models.

In [ ]:
tokenizer = processor.tokenizer

def get_token_ids(tokenizer, word):
    ids = set()
    for prefix in ['', ' ', '\u2581']:   # '' , space, SentencePiece leading space
        enc = tokenizer.encode(prefix + word, add_special_tokens=False)
        if len(enc) == 1:
            ids.add(enc[0])
    for candidate in [word, word.lower(), word.upper(), ' ' + word]:
        tok_id = tokenizer.convert_tokens_to_ids(candidate)
        if tok_id not in (tokenizer.unk_token_id, None):
            ids.add(tok_id)
    return sorted(ids)

YES_IDS = sorted(set(get_token_ids(tokenizer, 'Yes') + get_token_ids(tokenizer, 'yes')))
NO_IDS  = sorted(set(get_token_ids(tokenizer, 'No')  + get_token_ids(tokenizer, 'no')))

print(f'Yes token IDs : {YES_IDS}')
print(f'  decodes to  : {[tokenizer.decode([i]) for i in YES_IDS]}')
print(f'No  token IDs : {NO_IDS}')
print(f'  decodes to  : {[tokenizer.decode([i]) for i in NO_IDS]}')

assert len(YES_IDS) > 0 and len(NO_IDS) > 0, \
    'Could not resolve Yes/No token IDs — check tokenizer vocabulary.'

## Cell 9 — Inference Loop

For each closed question:
1. Build the v2 prompt: `'Answer the question with yes or no.\n\n{question}\n\nFinal Answer:'`
2. Run `model.generate()` with `output_scores=True`, `max_new_tokens=1`, `do_sample=False`.
3. `outputs.scores[0]` — logits at the first generated token, shape `(1, vocab_size)`.
4. Softmax → sum Yes-token probabilities → `p_yes_raw`; same for No.
5. Normalise: `p_yes = p_yes_raw / (p_yes_raw + p_no_raw)`.
6. Prediction: `'Yes'` if `p_yes_raw ≥ p_no_raw`, else `'No'`.

In [ ]:
def build_closed_prompt(question):
    return f'Answer the question with yes or no.\n\n{question}\n\nFinal Answer:'


def compute_p_yes(scores_0, yes_ids, no_ids):
    probs     = F.softmax(scores_0[0], dim=-1)    # (vocab_size,)
    p_yes_raw = sum(probs[i].item() for i in yes_ids if i < probs.shape[0])
    p_no_raw  = sum(probs[i].item() for i in no_ids  if i < probs.shape[0])
    denom     = p_yes_raw + p_no_raw
    p_yes_norm = p_yes_raw / denom if denom > 0 else 0.5
    return p_yes_raw, p_no_raw, p_yes_norm


# ── Resume support ─────────────────────────────────────────────────────────────
completed = {}
if os.path.exists(OUT_PATH):
    with open(OUT_PATH) as f:
        for line in f:
            try:
                r = json.loads(line)
                if r.get('p_yes_norm') is not None or 'error' in r:
                    completed[r['idx']] = r
            except:
                pass
    print(f'Resuming: {len(completed)} / {len(records)} already done.')

errors = 0
f_out  = open(OUT_PATH, 'a')

for i, sample in enumerate(tqdm(records, desc=RUN_CONFIG)):
    if i in completed:
        continue

    try:
        image    = get_image(sample)
        question = get_question(sample)
        gt       = get_ground_truth(sample)   # 'yes' or 'no'
        prompt   = build_closed_prompt(question)

        messages = [{
            'role': 'user',
            'content': [
                {'type': 'image'},
                {'type': 'text', 'text': prompt},
            ]
        }]
        text   = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = processor(text=text, images=image, return_tensors='pt').to(device)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1,
                output_scores=True,
                return_dict_in_generate=True,
                do_sample=False,
            )

        scores_0 = outputs.scores[0]   # (1, vocab_size)
        p_yes_raw, p_no_raw, p_yes_norm = compute_p_yes(scores_0, YES_IDS, NO_IDS)

        first_tok   = outputs.sequences[0, inputs['input_ids'].shape[-1]]
        raw_token   = tokenizer.decode([first_tok]).strip()
        prediction  = 'Yes' if p_yes_raw >= p_no_raw else 'No'
        correct     = (prediction.lower() == gt.lower())

        record = {
            'idx':          i,
            'question':     question,
            'ground_truth': gt,
            'prediction':   prediction,
            'raw_token':    raw_token,
            'p_yes_raw':    round(p_yes_raw,  6),
            'p_no_raw':     round(p_no_raw,   6),
            'p_yes_norm':   round(p_yes_norm,  6),
            'correct':      correct,
            'model':        MODEL_ID,
            'model_short':  MODEL_SHORT,
            'dataset':      DATASET,
        }

    except Exception as e:
        errors += 1
        print(f'  Error idx={i}: {e}')
        record = {
            'idx': i, 'question': '', 'ground_truth': '', 'prediction': '',
            'raw_token': '', 'p_yes_raw': None, 'p_no_raw': None, 'p_yes_norm': None,
            'correct': None, 'model': MODEL_ID, 'model_short': MODEL_SHORT,
            'dataset': DATASET, 'error': str(e),
        }

    f_out.write(json.dumps(record) + '\n')
    f_out.flush()

f_out.close()
print(f'\nDone. {len(records)} questions  |  {errors} errors  ->  {OUT_PATH}')

## Cell 10 — Quick Sanity Check

In [ ]:
valid = [json.loads(l) for l in open(OUT_PATH)
         if 'error' not in l and json.loads(l).get('p_yes_norm') is not None]

acc       = sum(1 for r in valid if r['correct']) / len(valid) if valid else 0
mean_conf = sum(r['p_yes_norm'] for r in valid) / len(valid)   if valid else 0
yes_recs  = [r for r in valid if r['ground_truth'] == 'yes']
no_recs   = [r for r in valid if r['ground_truth'] == 'no']

print(f'Run          : {RUN_CONFIG}')
print(f'N valid      : {len(valid)}')
print(f'Accuracy     : {acc*100:.2f}%')
print(f'Yes-acc      : {sum(1 for r in yes_recs if r["correct"])/len(yes_recs)*100:.2f}%  (N={len(yes_recs)})')
print(f'No-acc       : {sum(1 for r in no_recs  if r["correct"])/len(no_recs) *100:.2f}%  (N={len(no_recs)})')
print(f'Mean P(Yes)  : {mean_conf*100:.2f}%  (normalised)')
print()
print('Sample predictions:')
for r in valid[:6]:
    ck = '\u2713' if r['correct'] else '\u2717'
    print(f"  {ck} GT={r['ground_truth']:<4} Pred={r['prediction']:<4} "
          f"P(Yes)={r['p_yes_norm']:.3f}  raw_token='{r['raw_token']}'  "
          f"Q: {r['question'][:50]}")

## Cell 11 — Next Steps

1. Download the output `.jsonl` from the **Output** tab.
2. Place it in `outputs/_archive/calibration/` with its exact filename.
3. Run all 4 configs (A–D), then locally:
   ```bash
   python3 scripts/calibration_analysis.py
   ```